In [ ]:
!pip install kagglehub

# 흉부 X-ray 분류 실험

**질문:** 사전 학습이 작은 CNN 기준 모델보다 도움이 되는가? 같은 ResNet18에서 동결 범위를 바꾸면 성능은 어떻게 달라지는가?

이 노트북은 원본 데이터를 먼저 살펴보고, 반복 가능한 분할로 모델을 학습한 뒤 검증셋으로 비교한다. 제공된 테스트셋은 마지막에 선택된 모델만 평가한다. 학습·평가 코드는 `src/`에 분리했다.


# 데이터 불러오기

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import pandas as pd
import kagglehub
from torchvision import datasets
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('프로젝트 루트 또는 notebooks 폴더에서 실행하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import make_loaders
from src.model import create_model
from src.train import train_model
from src.evaluate import evaluate


In [ ]:
# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Dataset path:", path)

In [ ]:
# GPU 설정
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

In [ ]:
train_dir = os.path.join(path, "chest_xray", "train")
val_dir = os.path.join(path, "chest_xray", "val")
test_dir = os.path.join(path, "chest_xray", "test")

# transform 전에 원본 데이터 확인

In [ ]:
train_dataset = datasets.ImageFolder(train_dir)

In [ ]:
path, label = train_dataset.samples[0]

with Image.open(path) as img:
    print(path)
    print(img.size)  # (가로 픽셀, 세로 픽셀)

In [ ]:
sizes = Counter()
for path, _ in train_dataset.samples:
    with Image.open(path) as img:
        sizes[img.size] += 1

train_data_size_table = pd.DataFrame(
    [(width, height, count) for (width, height), count in sizes.items()],
    columns=["width", "height", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

In [ ]:
train_data_size_table.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(
    data=train_data_size_table,
    x="width",
    y="height",
    size="count",
    sizes=(20, 500),
    alpha=0.6,
    legend="brief",
    ax=axes[0],
)

axes[0].set_title("Training image sizes — scatter")
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Height (pixels)")

sns.histplot(
    data=train_data_size_table,
    x="width",
    y="height",
    weights="count",
    bins=40,
    cmap="Blues",
    cbar=True,
    cbar_kws={"label": "Number of images"},
    ax=axes[1],
)
axes[1].set_title("Training image sizes — histogram")
axes[1].set_xlabel("Width (pixels)")
axes[1].set_ylabel("Height (pixels)")

fig.tight_layout()
plt.show()

In [ ]:
class_counts = (
    pd.Series(train_dataset.targets)
    .map(dict(enumerate(train_dataset.classes)))
    .value_counts()
)

class_table = pd.DataFrame({
    "class": class_counts.index,
    "count": class_counts.values,
    "percent": class_counts.values / class_counts.sum() * 100
})

In [ ]:
ax = sns.barplot(data=class_table, x="class", y="count")

for bar, percent in zip(ax.patches, class_table["percent"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{percent:.1f}%",
        ha="center",
        va="bottom"
    )

ax.set(xlabel="Class", ylabel="Number of images", title="Training class distribution")
plt.tight_layout()
plt.show()

## 검증셋 재구성

원래 train과 작은 val을 합친 뒤 클래스별로 80:20 분리한다. 증강은 train에만 적용하며, 검증·테스트에는 동일한 기본 전처리를 적용한다. 분할 seed를 고정해 모델끼리 같은 샘플에서 비교한다. 환자 식별자가 확보되면 환자 단위 분리도 확인해야 한다.


In [ ]:
SEED = 42
BATCH_SIZE = 32
EPOCHS = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

# scratch 모델용 전처리
scratch_data = make_loaders(path, batch_size=BATCH_SIZE, seed=SEED, augment=False, pretrained=False)
class_names = scratch_data['classes']
print('classes:', class_names, 'sizes:', scratch_data['sizes'])
# 다수 클래스로만 예측했을 때의 기준 정확도
raw_train = datasets.ImageFolder(train_dir)
raw_val = datasets.ImageFolder(val_dir)
all_targets = raw_train.targets + raw_val.targets
majority_baseline = max(Counter(all_targets).values()) / len(all_targets)
print('majority-class baseline:', round(majority_baseline, 3))


## 작은 CNN 기준 모델

사전 학습 없이 훈련한 모델을 기준으로 삼는다. 검증셋에서 오류를 분석한 뒤 증강 또는 모델 구조 실험으로 이어간다.


In [ ]:
torch.manual_seed(SEED)
scratch_model = create_model('custom_cnn', num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
scratch_optimizer = torch.optim.Adam(scratch_model.parameters(), lr=1e-3)
scratch_history = train_model(scratch_model, scratch_data['train'], scratch_data['val'],
                              criterion, scratch_optimizer, device, epochs=EPOCHS)
scratch_val = evaluate(scratch_model, scratch_data['val'], criterion, device, class_names)
print(pd.DataFrame(scratch_val['report']).T)
print('confusion matrix:\n', scratch_val['confusion_matrix'])


## ResNet18 전이학습: Frozen / Partial / Full

모든 실험은 동일한 데이터 분할을 사용한다. 각 실험은 새 pretrained 가중치에서 시작하고, 해당 모델에서 학습 가능한 파라미터만 옵티마이저에 전달한다. pretrained 가중치에 맞춰 3채널과 ImageNet 정규화를 적용한다.


In [ ]:
pretrained_data = make_loaders(path, batch_size=BATCH_SIZE, seed=SEED,
                               augment=False, pretrained=True)
assert pretrained_data['classes'] == class_names
experiments = {}
for mode, lr in [('frozen', 1e-3), ('partial', 1e-4), ('full', 1e-5)]:
    torch.manual_seed(SEED)
    net = create_model('resnet18', num_classes=len(class_names), pretrained=True, mode=mode).to(device)
    optimizer = torch.optim.Adam((p for p in net.parameters() if p.requires_grad), lr=lr)
    history = train_model(net, pretrained_data['train'], pretrained_data['val'],
                          criterion, optimizer, device, epochs=EPOCHS)
    metrics = evaluate(net, pretrained_data['val'], criterion, device, class_names)
    experiments[mode] = {'model': net, 'history': history, 'validation': metrics}
    print(mode, 'validation accuracy:', round(metrics['accuracy'], 4))


## 검증 성능 비교와 최종 모델 선택

모델 선택에는 검증 결과만 사용한다. 정확도와 폐렴 클래스의 precision, recall, F1을 함께 본다. 테스트셋은 아래에서 선택된 모델에 대해 한 번 평가한다.


In [ ]:
positive_class = 'PNEUMONIA'
rows = []
for mode, result in experiments.items():
    m = result['validation']
    p = m['report'][positive_class]
    rows.append({'model': mode, 'accuracy': m['accuracy'], 'precision': p['precision'],
                 'recall': p['recall'], 'f1': p['f1-score']})
comparison = pd.DataFrame(rows).set_index('model').sort_values('f1', ascending=False)
display(comparison)
best_mode = comparison.index[0]
print('선택한 모델:', best_mode)


In [ ]:
final_model = experiments[best_mode]['model']
test_metrics = evaluate(final_model, pretrained_data['test'], criterion, device, class_names)
print('test accuracy:', round(test_metrics['accuracy'], 4))
display(pd.DataFrame(test_metrics['report']).T)
print('confusion matrix:\n', test_metrics['confusion_matrix'])


## 다음 실험

1. 검증셋의 확신도 높은 오답을 확인하고 오류 유형을 기록한다.
2. AlexNet·VGG·GoogLeNet·ResNet 직접 구현 모델을 `create_model(name, num_classes=2, pretrained=False)`로 생성해 **같은 분할과 평가 함수**로 비교한다.
3. 약한 증강을 하나씩 적용해 검증 성능 변화와 오답 변화를 본다.
4. 가장 유망한 모델에만 Optuna를 적용한다. Optuna 역시 검증셋만 사용한다.

**결과 해석:** 위 표를 실행한 뒤 성능 차이, 오답 유형, 최종 모델을 고른 이유를 여기에 적는다.
